In [2]:
import numpy as np

In [12]:
!python -m numpy.f2py -c Diff_PH.f90 -m my_diffusion

Cannot use distutils backend with Python>=3.12, using meson backend instead.
Using meson backend
Will pass --lower to f2py
See https://numpy.org/doc/stable/f2py/buildtools/meson.html
Reading fortran codes...
	Reading file 'Diff_PH.f90' (format:free)
Post-processing...
	Block: my_diffusion
			Block: diffusion_ph
			Block: diffus
			Block: c_defect
			Block: nm_prog
			Block: f_impl
Applying post-processing hooks...
  character_backward_compatibility_hook
Post-processing (stage 2)...
Building modules...
    Building module "my_diffusion"...
    Generating possibly empty wrappers"
    Maybe empty "my_diffusion-f2pywrappers.f"
        Constructing wrapper function "diffus"...
          diffus(c_ph,x,n_t,[nc,nd])
    Generating possibly empty wrappers"
    Maybe empty "my_diffusion-f2pywrappers.f"
        Constructing wrapper function "c_defect"...
          c_defect(y,x,nj,[n])
    Generating possibly empty wrappers"
    Maybe empty "my_diffusion-f2pywrappers.f"
        Constructing wrappe

In [8]:
import my_diffusion
import numpy as np

my_diffusion.da3.x0 = 0.0
my_diffusion.da3.xl = 200.0
my_diffusion.da3.t0 = 0.0
my_diffusion.da3.tfin = 30.0

my_diffusion.da1.n_impl = 501
my_diffusion.da1.eps_c = 1.0e-3
my_diffusion.da1.itmax = 10000

my_diffusion.def_diff.pli = 10.0
my_diffusion.def_diff.gim = 2.7e4
my_diffusion.def_diff.rp = 0.0108
my_diffusion.def_diff.drp = 0.0068
my_diffusion.def_diff.v0 = 200.0
my_diffusion.def_diff.al1 = 1.0
my_diffusion.def_diff.beta1 = -1.0e-3
my_diffusion.def_diff.ga1 = 0.0
my_diffusion.def_diff.al2 = 1.0
my_diffusion.def_diff.beta2 = -1.0
my_diffusion.def_diff.ga2 = 0.0
my_diffusion.def_diff.rpdrp = 0.015
my_diffusion.def_diff.x_star = 0.02

my_diffusion.ph_diff.en = 1000.0
my_diffusion.ph_diff.eni = 1.0e-12
my_diffusion.ph_diff.bf1 = 4.0
my_diffusion.ph_diff.bf2 = 0.1
my_diffusion.ph_diff.dfi = 1.5e-7
my_diffusion.ph_diff.af = 1.0
my_diffusion.ph_diff.be1 = 0.0076
my_diffusion.ph_diff.be2 = 0.0001
my_diffusion.ph_diff.dei = 4.0e-7
my_diffusion.ph_diff.ae = 1.0

nc = 500
nd = 200000
n_t = 3000
c_ph = np.zeros(nc + 1)
x = np.zeros(nd + 1)

try:
    print("Запуск расчета...")
    my_diffusion.diffus(c_ph, x, n_t)
    print("Расчет завершен!")
    print("Результат (первые 5):", c_ph[:5])
except Exception as e:
    print("Произошла ошибка:", e)

Запуск расчета...
Расчет завершен!
Результат (первые 5): [0. 0. 0. 0. 0.]
 STOP -- itmax


In [2]:
import subprocess
import os

exe_path = "./Diff_PH.exe"

if not os.path.exists(exe_path):
    print(f"Ошибка: Файл {exe_path} не найден в {os.getcwd()}")
else:
    try:
        result = subprocess.run([exe_path], capture_output=True, text=True)
        print(result.stdout)
    except PermissionError:
        print("Ошибка: Файл найден, но нет прав на исполнение. Выполните 'chmod +x Diff_PH.exe'")
    except Exception as e:
        print(f"Произошла ошибка: {e}")

Ошибка: Файл ./Diff_PH.exe не найден в /mnt/d/gpwork/test


In [1]:
import math
import sys

# Global Variables (to mimic Fortran COMMON blocks)
# COMMON /DA1/ NC1,ND1,N_impl,H,TAU,eps_C,itmax
NC1 = 0
ND1 = 0
N_impl_val = 0
H = 0.0
TAU = 0.0
eps_C = 0.0
itmax = 0

# COMMON /PH_DIFF/ EN,ENI,BF1,BF2,DFI,AF,BE1,BE2,DEI,AE
EN = 0.0
ENI = 0.0
BF1 = 0.0
BF2 = 0.0
DFI = 0.0
AF = 0.0
BE1 = 0.0
BE2 = 0.0
DEI = 0.0
AE = 0.0

# COMMON /DA3/X0,XL,T0,TFIN
X0 = 0.0
XL = 0.0
T0 = 0.0
TFIN = 0.0

# COMMON /DEF_DIFF/PLI,GIM,RP,DRP,V0,al1,beta1,ga1,al2,beta2,ga2,RPDRP,X_STAR
PLI = 0.0
GIM = 0.0
RP = 0.0
DRP = 0.0
V0 = 0.0
al1 = 0.0
beta1 = 0.0
ga1 = 0.0
al2 = 0.0
beta2 = 0.0
ga2 = 0.0
RPDRP = 0.0
X_STAR = 0.0

def NM_PROG(A, B, C, F, Y, N, NJ):
    """
    Subroutine for solving a system of equations using a specific 
    sweep method (variation of Thomas algorithm).
    """
    # A(1:N), B(0:NJ), C(0:N), F(0:N), Y(0:N)
    ITETA = [0] * (N + 1)
    ICAPA = [0] * (N + 1)
    AL = [0.0] * (N + 1)
    BE = [0.0] * (N + 1)

    ICAPA[0] = 0
    CC = C[0]
    FF = F[0]
    FI = F[1]
    AA = A[1]

    for I in range(NJ + 1):
        # Implementation of the conditional logic (GOTO 51/53)
        if abs(CC) >= abs(B[I]):
            # Label 51
            AL[I + 1] = B[I] / CC
            BE[I + 1] = FF / CC
            CC = C[I + 1] - AA * AL[I + 1]
            FF = FI + AA * BE[I + 1]
            ITETA[I + 1] = ICAPA[I]
            ICAPA[I + 1] = I + 1
            if I != NJ:
                AA = A[I + 2]
                FI = F[I + 2]
        else:
            # Label 53
            AL[I + 1] = CC / B[I]
            BE[I + 1] = -FF / B[I]
            CC = C[I + 1] * AL[I + 1] - AA
            FF = FI - C[I + 1] * BE[I + 1]
            ITETA[I + 1] = I + 1
            ICAPA[I + 1] = ICAPA[I]
            if I != NJ:
                AA = A[I + 2] * AL[I + 1]
                FI = F[I + 2] + A[I + 2] * BE[I + 1]
    
    # Label 55 (End of Loop)
    NN = ICAPA[N]
    Y[NN] = FF / CC
    
    I = NJ
    while I >= 0:
        # Label 66
        MM = ITETA[I + 1]
        NN = ICAPA[I + 1]
        Y[MM] = AL[I + 1] * Y[NN] + BE[I + 1]
        I -= 1

def F_IMPL(T, X, CT):
    """
    Interpolation function for initial implantation profile.
    T: target point, X: array of depth, CT: array of concentration
    """
    global N_impl_val
    res = 0.0
    for I in range(N_impl_val - 1): # 1 to N_impl-1 in Fortran
        # Adjust indices for 0-based Python (Fortran 1..N_impl -> Python 0..N_impl-1)
        if T >= X[I] and T <= X[I+1]:
            res = CT[I] + (CT[I+1] - CT[I]) * (T - X[I]) / (X[I+1] - X[I])
            # Note: Fortran doesn't break, but mathematically only one segment matches
    
    if T <= X[0]:
        res = CT[0]
    if T >= X[N_impl_val - 1]:
        res = CT[N_impl_val - 1]
    return res

def C_DEFECT(Y, X, N, NJ):
    """
    Subroutine to calculate defect distribution.
    """
    global PLI, GIM, RP, DRP, V0, al1, beta1, ga1, al2, beta2, ga2, RPDRP, X_STAR
    global NC1, ND1, N_impl_val, H, TAU, eps_C, itmax

    print(f"V0=SUBR {V0}")
    NC_local = NC1 + 1
    
    V = [0.0] * (N + 1)
    G = [0.0] * (N + 1)
    
    # Open V.dat
    with open('V.dat', 'w') as f33:
        for J in range(N + 1):
            if X[J] < X_STAR:
                V[J] = V0
            else:
                V[J] = V0 * math.exp(-(X[J] - X_STAR)**2 / (2.0 * RPDRP**2))
        
        for J in range(NC_local):
            f33.write(f" {X[J]:16.9E} {V[J]:16.9E}\n")
            
    Q = 1.0 / PLI**2
    print(f"N={N}")
    
    # Open G(X).dat
    with open('G(X).dat', 'w') as f21:
        for J in range(N + 1):
            if X[J] < X_STAR:
                G[J] = 1.0 + GIM
            else:
                G[J] = 1.0 + GIM * math.exp(-(X[J] - X_STAR)**2 / (2.0 * RPDRP**2))
        
        for J in range(NC_local):
            f21.write(f" {X[J]:16.9E} {G[J]:16.9E}\n")

    F0 = -G[0] / PLI**2
    FN = -G[N] / PLI**2
    
    F = [0.0] * (N + 1)
    for J in range(1, NJ + 1):
        F[J] = H**2 * G[J] / PLI**2
        
    PP = ga1 + al1 * H + H**2 * Q * ga1 * 0.5
    CAP1 = ga1 / PP
    PMU1 = -(beta1 * H + 0.5 * H**2 * ga1 * F0) / PP
    DD = ga2 + al2 * H + H**2 * Q * ga2 * 0.5
    PMU2 = (-0.5 * H**2 * ga2 * FN - beta2 * H) / DD
    CAP2 = ga2 / DD
    
    C = [0.0] * (N + 1)
    A = [0.0] * (N + 1)
    B = [0.0] * (NJ + 1)
    
    # Initialization
    C[0] = 1.0
    for J in range(1, NJ + 1):
        C[J] = H**2 * Q + 2.0
    C[N] = 1.0
    
    for J in range(1, NJ + 1):
        A[J] = 1.0 + H * V[J-1] / 2.0
    A[N] = CAP2
    
    B[0] = CAP1
    for J in range(1, NJ + 1):
        B[J] = 1.0 - H * V[J+1] / 2.0
        
    F[0] = PMU1
    F[N] = PMU2
    
    NM_PROG(A, B, C, F, Y, N, NJ)

def DIFFUS(C_PH, X, NC, ND, N_T):
    global NC1, ND1, N_impl_val, H, TAU, eps_C, itmax
    global EN, ENI, BF1, BF2, DFI, AF, BE1, BE2, DEI, AE
    global X0, XL, T0, TFIN
    global PLI, GIM, RP, DRP, V0, al1, beta1, ga1, al2, beta2, ga2, RPDRP, X_STAR

    C = [0.0] * (NC + 1)
    A = [0.0] * (NC + 1) # 1:NC
    B = [0.0] * (NC1 + 1) # 0:NC1
    F = [0.0] * (NC + 1)
    A1 = [0.0] * (NC + 1)
    A2 = [0.0] * (NC + 1)
    A3 = [0.0] * (NC + 1)
    KE1 = [0.0] * (NC + 1)
    KF1 = [0.0] * (NC + 1)
    
    CN_PH = [0.0] * (NC + 1)
    Y = [0.0] * (NC + 1)
    YS = [0.0] * (NC + 1)
    C_T0 = [0.0] * (NC + 1)
    C_V = [0.0] * (ND + 1)
    C_I = [0.0] * (ND + 1)
    STOK = [0.0] * (NC + 1)
    
    X_IMPL = [0.0] * N_impl_val
    C_IMPL = [0.0] * N_impl_val
    
    f_integral = open('INTEGRAL.dat', 'w')
    
    # Read ph-impl.dat
    try:
        with open('ph-impl.dat', 'r') as f7:
            for i in range(N_impl_val):
                line = f7.readline()
                if not line: break
                parts = line.split()
                X_IMPL[i] = float(parts[0])
                C_IMPL[i] = float(parts[1]) * 1.0E-12
    except FileNotFoundError:
        print("Error: ph-impl.dat not found.")
        sys.exit(1)

    H = (XL - X0) / ND
    TAU = (TFIN - T0) / N_T
    print(f"H={H}    TAU={TAU}")
    
    X[0] = X0
    for J in range(1, ND):
        X[J] = X[J-1] + H
    X[ND] = XL
    
    with open('C_T0.dat', 'w') as f15:
        for I in range(NC + 1):
            C_T0[I] = F_IMPL(X[I], X_IMPL, C_IMPL)
            f15.write(f" {X[I]:16.7E} {C_T0[I]*1.0E12:16.7E}\n")
            
    for I in range(ND + 1):
        C_V[I] = 1.0
        C_I[I] = 1.0
        
    C_DEFECT(C_I, X, ND, ND1)
    
    with open('C_I.dat', 'w') as f6:
        for I in range(NC + 1):
            f6.write(f" {X[I]:16.7E} {C_I[I]:16.7E}\n")
            
    # Modify for C_V
    saved_beta1 = beta1
    saved_V0 = V0
    beta1 = -10.0
    V0 = 0.0
    C_DEFECT(C_V, X, ND, ND1)
    
    with open('C_V.dat', 'w') as f11:
        for I in range(NC + 1):
            f11.write(f" {X[I]:16.7E} {C_V[I]:16.7E}\n")
            
    # Restore
    beta1 = saved_beta1
    V0 = saved_V0
    
    T_curr = T0
    KSLOI = 0
    print(f"KSLOI_0={KSLOI}")
    
    for I in range(NC + 1):
        CN_PH[I] = C_T0[I]
        STOK[I] = 0.0
        
    # Main Time Loop
    # Label 505
    while True:
        T_curr = T_curr + TAU
        KSLOI = KSLOI + 1
        print(f"KSLOI={KSLOI}")
        
        for I in range(NC + 1):
            YS[I] = CN_PH[I]
            
        iter_C = 1
        # Label 555
        while True:
            for I in range(NC + 1):
                # CAPA Calculation
                val_diff = YS[I] - EN
                sqrt_val = math.sqrt(val_diff**2 + 4.0 * ENI**2)
                CAPA = (val_diff + sqrt_val) / (2.0 * ENI)
                
                KE1[I] = AE * DEI * ((1.0 + BE1 * CAPA + BE2 * CAPA**2) / (1.0 + BE1 + BE2))
                KF1[I] = AF * DFI * ((1.0 + BF1 * CAPA + BF2 * CAPA**2) / (1.0 + BF1 + BF2))
                
            for I in range(1, NC + 1):
                A1[I] = 0.5 * (KE1[I] + KE1[I-1])
                A2[I] = 0.5 * (KF1[I] + KF1[I-1])
                
                SQ = math.sqrt((YS[I] - EN)**2 + 4.0 * ENI**2)
                SQ1 = math.sqrt((YS[I-1] - EN)**2 + 4.0 * ENI**2)
                
                A_K = (KE1[I] * YS[I] * C_V[I] + KF1[I] * YS[I] * C_I[I]) / SQ
                A1_K = (KE1[I-1] * YS[I-1] * C_V[I-1] + KF1[I-1] * YS[I-1] * C_I[I-1]) / SQ1
                A3[I] = 0.5 * (A_K + A1_K)
                
            for I in range(1, NC + 1):
                A[I] = A1[I] * C_V[I-1] + A2[I] * C_I[I-1] + A3[I]
            
            for I in range(NC1 + 1):
                B[I] = A1[I+1] * C_V[I+1] + A2[I+1] * C_I[I+1] + A3[I+1]
                
            C_arr = [0.0] * (NC + 1)
            C_arr[0] = 0.5 * H**2 / TAU + A[1]
            for I in range(1, NC1 + 1):
                C_arr[I] = H**2 / TAU + A[I+1] + B[I-1]
            C_arr[NC] = 0.5 * H**2 / TAU + B[NC1]
            
            F_arr = [0.0] * (NC + 1)
            F_arr[0] = 0.5 * H**2 * CN_PH[0] / TAU
            for I in range(1, NC1 + 1):
                F_arr[I] = H**2 * CN_PH[I] / TAU + TAU * H * STOK[I]
            F_arr[NC] = 0.5 * H**2 * CN_PH[NC] / TAU
            
            NM_PROG(A, B, C_arr, F_arr, Y, NC, NC1)
            
            # Convergence check
            converged = True
            for I in range(NC1 + 1):
                if abs(YS[I]) > 1.0:
                    eps1 = eps_C
                    eps2 = 0.0
                else:
                    eps1 = 0.0
                    eps2 = eps_C
                
                if abs(Y[I] - YS[I]) >= (eps1 * abs(YS[I]) + eps2):
                    converged = False
                    break
            
            if converged and iter_C > 1:
                # Label 666 branch
                break
            
            # Label 632 logic
            for I in range(NC + 1):
                YS[I] = Y[I]
            
            if iter_C >= itmax:
                print(" STOP -- iter_C.GT.itmax -- ")
                # Go to Label 55 (End)
                f_integral.close()
                return
                
            iter_C += 1
            # Continue Label 555
            
        # Label 666
        print(f"GO TO NEW LAY:  iter_C={iter_C}")
        S1 = 0.0
        for I in range(NC + 1):
            SI = Y[I]
            if I == 0 or I == NC:
                SI = SI / 2.0
            S1 += SI
        S1 = S1 * H
        
        f_integral.write(f" {T_curr:16.7E} {S1:16.7E}\n")
        print(f"Time={T_curr}   Integral_S1={S1}")
        
        if KSLOI >= N_T:
            # Label 266
            break
        
        for I in range(NC + 1):
            CN_PH[I] = Y[I]
        # Continue Label 505
        
    # Label 266
    for I in range(NC + 1):
        C_PH[I] = Y[I]
        
    f_integral.close()
    print(f"H={H}    TAU={TAU}    Time={T_curr}  eps_C={eps_C}")
    print(f"STOK(I)={STOK[1]}")

def main():
    global NC1, ND1, N_impl_val, H, TAU, eps_C, itmax
    global EN, ENI, BF1, BF2, DFI, AF, BE1, BE2, DEI, AE
    global X0, XL, T0, TFIN
    global PLI, GIM, RP, DRP, V0, al1, beta1, ga1, al2, beta2, ga2, RPDRP, X_STAR

    ND = 200000
    NC = 500
    N_T = 3000
    
    C_PH = [0.0] * (NC + 1)
    X = [0.0] * (ND + 1)
    
    # Initialize Data
    PLI = 10.0
    GIM = 2.7E4
    RP = 0.0108
    DRP = 0.0068
    RPDRP = 0.015
    X_STAR = 0.02
    al1 = 1.0
    beta1 = -1.0E-3
    ga1 = 0.0
    al2 = 1.0
    beta2 = -1.0
    ga2 = 0.0
    V0 = 200.0
    
    N_impl_val = 501
    eps_C = 1.0E-3
    itmax = 10000
    
    BF1 = 4.0
    BF2 = 0.1
    DFI = 1.5E-7
    AF = 1.0
    BE1 = 0.0076
    BE2 = 0.0001
    DEI = 4.0E-7
    AE = 1.0
    
    TE = 900.0
    EKB = 8.61709715681519E-5
    TE_K = TE + 273.15
    EN = 1.0E3
    ENIE = 0.605
    ENIF = 1.5
    ENIO = 3.87298E16
    ENI = 1.0E-12 * ENIO * math.exp(-ENIE / (EKB * TE_K)) * (TE_K**ENIF)
    C_enh = 1.0E-12 * 1.65E23 * math.exp(-0.88 / (EKB * TE_K))
    
    X0 = 0.0
    XC = 0.5
    XL = 200.0
    T0 = 0.0
    TFIN = 30.0
    NC1 = NC - 1
    ND1 = ND - 1
    
    DIFFUS(C_PH, X, NC, ND, N_T)
    
    # Output
    with open('C_PH.dat', 'w') as f3:
        for I in range(NC + 1):
            f3.write(f" {X[I]:16.7E} {C_PH[I]*1.0E12:16.7E}\n")
            
    print(f"C_enh={C_enh}")
    print(f"eni={ENI}")

if __name__ == "__main__":
    main()



H=0.001    TAU=0.01
V0=SUBR 200.0
N=200000
V0=SUBR 0.0
N=200000
KSLOI_0=0
KSLOI=1
GO TO NEW LAY:  iter_C=5
Time=0.01   Integral_S1=49823200.43583376
KSLOI=2
GO TO NEW LAY:  iter_C=5
Time=0.02   Integral_S1=49823200.43583375
KSLOI=3
GO TO NEW LAY:  iter_C=5
Time=0.03   Integral_S1=49823200.43583379
KSLOI=4
GO TO NEW LAY:  iter_C=5
Time=0.04   Integral_S1=49823200.435833745
KSLOI=5
GO TO NEW LAY:  iter_C=5
Time=0.05   Integral_S1=49823200.43583377
KSLOI=6
GO TO NEW LAY:  iter_C=5
Time=0.060000000000000005   Integral_S1=49823200.435833745
KSLOI=7
GO TO NEW LAY:  iter_C=5
Time=0.07   Integral_S1=49823200.43583374
KSLOI=8
GO TO NEW LAY:  iter_C=5
Time=0.08   Integral_S1=49823200.435833745
KSLOI=9
GO TO NEW LAY:  iter_C=5
Time=0.09   Integral_S1=49823200.43583372
KSLOI=10
GO TO NEW LAY:  iter_C=5
Time=0.09999999999999999   Integral_S1=49823200.43583371
KSLOI=11
GO TO NEW LAY:  iter_C=5
Time=0.10999999999999999   Integral_S1=49823200.43583366
KSLOI=12
GO TO NEW LAY:  iter_C=5
Time=0.119999999